# P01 — CPA training across all conditions and seeds

Trains the Compositional Perturbation Autoencoder (CPA) across the four conditions (K562 random, K562 stratified, RPE1 random, RPE1 stratified) and the seed range configured by environment variable.

**Manuscript reference:** Methods §2.2 (Model and training protocol).

**Run modes:**
- `QUICK_DEMO=1` — 2 seeds per condition (8 runs total), approximately 30 minutes on Apple Silicon CPU.
- `FULL_TRAINING=1` — 100 seeds per condition (400 runs total), approximately 13 hours on Apple Silicon CPU.
- Neither set — no work performed; prints a usage note and exits.

**Inputs:**
- `data/replogle/{k562,rpe1}_essential.h5ad` (from D01).
- `data/holdout_specs/holdout_specs.csv` (from D03).
- `data/severity_refs/replogle_{K562,RPE1}_severity.csv` (from D02).

**Outputs (one per condition × seed):**
- `data/predictions/severity_details/CPA_{cell_type}_{split_type}_seed{seed}.severity.h5ad`

Each output h5ad has `.obs` columns `predicted_mean_abs_delta`, `leverage_score`, and `perturbation_target`, ready for consumption by P03 and P04.

**Dependencies:** this notebook requires the `cpa` dependency group:
```bash
uv sync --group cpa
```


In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from perturb_style import DATA_DIR

# Run-mode environment variables (exported by run_all.sh)
QUICK_DEMO = os.environ.get("QUICK_DEMO", "0") == "1"
FULL_TRAINING = os.environ.get("FULL_TRAINING", "0") == "1"

if QUICK_DEMO:
    SEEDS = [1, 2]
    print("Mode: QUICK_DEMO  (2 seeds per condition; 8 runs total; ~30 min)")
elif FULL_TRAINING:
    SEEDS = list(range(1, 101))
    print("Mode: FULL_TRAINING  (100 seeds per condition; 400 runs total; ~13 hours)")
else:
    SEEDS = []
    print("No training mode set. This notebook performs no work without")
    print("QUICK_DEMO=1 or FULL_TRAINING=1 (set by ./run_all.sh --demo or --train-full).")
    print()
    print("If you only need to reproduce the figures and tables, run:")
    print("  ./run_all.sh --figures")


Mode: QUICK_DEMO  (2 seeds per condition; 8 runs total; ~30 min)


## Check prerequisites

Verify that the inputs from D01, D02, D03 are present before launching training. Exit gracefully with a helpful message if any are missing.

In [2]:
ATLAS_PATHS = {
    "K562": DATA_DIR / "replogle" / "k562_essential.h5ad",
    "RPE1": DATA_DIR / "replogle" / "rpe1_essential.h5ad",
}
SPEC_PATH = DATA_DIR / "holdout_specs" / "holdout_specs.csv"
SEV_REFS = {
    "K562": DATA_DIR / "severity_refs" / "replogle_K562_severity.csv",
    "RPE1": DATA_DIR / "severity_refs" / "replogle_RPE1_severity.csv",
}
OUT_DIR = DATA_DIR / "predictions" / "severity_details"
OUT_DIR.mkdir(exist_ok=True, parents=True)

prerequisites_ok = SEEDS and all(p.exists() for p in [*ATLAS_PATHS.values(), SPEC_PATH, *SEV_REFS.values()])

if SEEDS and not prerequisites_ok:
    print("Training mode set but prerequisites are missing:")
    for name, path in [
        ("K562 atlas", ATLAS_PATHS["K562"]),
        ("RPE1 atlas", ATLAS_PATHS["RPE1"]),
        ("Holdout specs", SPEC_PATH),
        ("K562 severity", SEV_REFS["K562"]),
        ("RPE1 severity", SEV_REFS["RPE1"]),
    ]:
        marker = "OK" if path.exists() else "MISSING"
        print(f"  [{marker}] {name}: {path}")
    print()
    print("Run D01 (atlases), D02 (severity references), and D03 (holdout specs) first:")
    print("  ./run_all.sh --demo        # also runs P01-P04 + figures after D-series")
    print("  ./run_all.sh --train-full  # same, with full sweep")
elif SEEDS:
    print("All prerequisites present:")
    for name, path in [
        ("K562 atlas", ATLAS_PATHS["K562"]),
        ("RPE1 atlas", ATLAS_PATHS["RPE1"]),
        ("Holdout specs", SPEC_PATH),
        ("K562 severity", SEV_REFS["K562"]),
        ("RPE1 severity", SEV_REFS["RPE1"]),
    ]:
        print(f"  [OK] {name}: {path}")


Training mode set but prerequisites are missing:
  [MISSING] K562 atlas: data/replogle/k562_essential.h5ad
  [MISSING] RPE1 atlas: data/replogle/rpe1_essential.h5ad
  [MISSING] Holdout specs: data/holdout_specs/holdout_specs.csv
  [MISSING] K562 severity: data/severity_refs/replogle_K562_severity.csv
  [MISSING] RPE1 severity: data/severity_refs/replogle_RPE1_severity.csv

Run D01 (atlases), D02 (severity references), and D03 (holdout specs) first:
  ./run_all.sh --demo        # also runs P01-P04 + figures after D-series
  ./run_all.sh --train-full  # same, with full sweep


## Train CPA per condition × seed

For each (cell_type, split_type, seed), train CPA from scratch on the 150-perturbation pool minus the 30-perturbation holdout, then predict on the held-out perturbations and compute the per-perturbation predicted_mean_abs_delta. Write the severity-detail h5ad consumed by P03 and P04.

**Implementation note.** The substantive CPA training and per-perturbation evaluation loop is delegated to `src/perturb_eval/train_cpa.py`, which wraps the CPA API. That module is currently a documented stub; the manuscript's reported runs were produced by the original `a_perturb_AI` orchestrator (`tools/run_calibration.py` and its descendants), which this notebook will eventually mirror. Until that mirror is in place, the cell below documents the loop structure and exits without training.


In [3]:
if SEEDS and prerequisites_ok:
    from itertools import product

    CONDITIONS = [(c, s) for c in ("K562", "RPE1") for s in ("random", "stratified")]
    total = len(CONDITIONS) * len(SEEDS)
    print(f"Plan: {total} CPA training runs ({len(CONDITIONS)} conditions x {len(SEEDS)} seeds).")
    print()

    try:
        from perturb_eval.train_cpa import train_one_seed
    except ImportError:
        train_one_seed = None

    if train_one_seed is None:
        print("perturb_eval.train_cpa.train_one_seed is not yet implemented.")
        print("This notebook is the orchestration shell; the substantive training")
        print("loop will be filled in alongside the module stub. For the manuscript's")
        print("reported runs, see a_perturb_AI/tools/run_calibration.py.")
        print()
        print("Planned per-seed flow:")
        print("  1. Read atlas h5ad for cell_type.")
        print("  2. Look up holdout perturbations for (cell_type, split_type, seed) from D03 spec.")
        print("  3. Subset atlas to (150-perturbation pool minus 30-perturbation holdout) for training.")
        print("  4. Train CPA with seed=seed for 30 epochs, batch_size=128, lr=1e-3, Adam.")
        print("  5. Predict per-cell post-perturbation expression on the 30-perturbation holdout.")
        print("  6. Aggregate to per-perturbation predicted_mean_abs_delta.")
        print("  7. Attach leverage_score column from D02 severity reference.")
        print("  8. Write data/predictions/severity_details/CPA_{cell}_{split}_seed{N}.severity.h5ad.")
    else:
        ok, missing = 0, []
        for (cell, split), seed in product(CONDITIONS, SEEDS):
            try:
                path = train_one_seed(cell_type=cell, split_type=split, seed=seed,
                                       atlas_path=ATLAS_PATHS[cell],
                                       spec_path=SPEC_PATH,
                                       severity_ref_path=SEV_REFS[cell],
                                       output_dir=OUT_DIR)
                ok += 1
                print(f"  [{ok}/{total}] CPA {cell} {split} seed{seed} -> {path.name}")
            except Exception as e:
                missing.append((cell, split, seed, str(e)))
                print(f"  FAIL CPA {cell} {split} seed{seed}: {e}")

        print(f"\nCompleted: {ok}/{total} runs successful; {len(missing)} failed.")
